# Archived research notebook

Original filename: `预测结果npy读取.ipynb`  
Purpose: Read and analyze saved prediction-result files.  
Historical research version retained for traceability. The original test data are unavailable; legacy outputs, execution state, and attachments were removed for public release.


In [ ]:
import numpy as np
import pandas as pd
import os
import matplotlib.pyplot as plt
from tabulate import tabulate
# import argparse # 移除了 argparse 模块导入

# -----------------------------------------------------------------------------
# IMPORTANT: This script is ONLY for analyzing the prediction output files
# generated by the 'predict' command of the main training/prediction script.
# It does NOT load or process original EEG data.
# -----------------------------------------------------------------------------

def load_prediction_results(output_dir):
    """
    加载预测结果文件并分析。
    尝试从指定的输出目录加载 predictions.npy, subject_predictions.npy 和 detailed_predictions.csv 文件。
    """
    # 文件路径 - 这些是预测命令生成的输出文件
    predictions_file = os.path.join(output_dir, 'predictions.npy')
    subject_predictions_file = os.path.join(output_dir, 'subject_predictions.npy')
    detailed_csv_file = os.path.join(output_dir, 'detailed_predictions.csv')
    
    results = {}
    
    print(f"尝试从目录加载预测结果文件：{output_dir}\n")

    # 加载文件级预测结果 (predictions.npy)
    # 这个文件通常包含每个输入的 EEG 片段（可能是单个文件或文件中的一个时间窗口）的预测结果和概率。
    if os.path.exists(predictions_file):
        try:
            # 使用 allow_pickle=True 加载 numpy 对象，因为保存的是字典结构
            file_results = np.load(predictions_file, allow_pickle=True).item()
            results['file_results'] = file_results
            print(f"✓ 成功加载文件预测结果: {predictions_file}")
            # 简单检查关键数据是否存在
            if 'file_names' not in file_results or 'predictions' not in file_results or 'probabilities' not in file_results:
                 print("警告: 文件预测结果文件 (predictions.npy) 可能不完整，缺少关键键 (file_names, predictions, 或 probabilities)。")
        except Exception as e:
            print(f"✗ 加载文件 {predictions_file} 时出错: {e}")
    else:
        print(f"✗ 文件不存在: {predictions_file}")
    
    # 加载受试者级预测结果 (subject_predictions.npy)
    # 这个文件通常包含按受试者ID汇总的预测结果，例如每个受试者有多少片段被预测为阳性，以及受试者级别的最终诊断（基于投票或平均概率）。
    if os.path.exists(subject_predictions_file):
        try:
             # 使用 allow_pickle=True 加载 numpy 对象，因为保存的是字典结构
            subject_results = np.load(subject_predictions_file, allow_pickle=True).item()
            results['subject_results'] = subject_results
            print(f"✓ 成功加载受试者预测结果: {subject_predictions_file}")
            # 简单检查是否包含任何受试者数据
            if not subject_results:
                 print("警告: 受试者预测结果文件 (subject_predictions.npy) 为空。")
        except Exception as e:
            print(f"✗ 加载文件 {subject_predictions_file} 时出错: {e}")
    else:
        print(f"✗ 文件不存在: {subject_predictions_file}")
    
    # 加载CSV详细预测 (detailed_predictions.csv)
    # 这个文件通常包含每个预测片段的详细信息，包括受试者ID、文件名、预测结果和概率，通常是表格形式，便于在电子表格软件中查看。
    if os.path.exists(detailed_csv_file):
        try:
            # 使用 pandas 读取 CSV
            csv_data = pd.read_csv(detailed_csv_file)
            results['csv_data'] = csv_data
            print(f"✓ 成功加载详细CSV数据: {detailed_csv_file}")
            if csv_data.empty:
                 print("警告: 详细CSV数据文件 (detailed_predictions.csv) 为空。")
        except Exception as e:
            print(f"✗ 加载文件 {detailed_csv_file} 时出错: {e}")
    else:
        print(f"✗ 文件不存在: {detailed_csv_file}")
    
    return results

def analyze_file_results(file_results):
    """
    分析并解读文件级预测结果。
    文件级结果是对每个单独输入的EEG片段的预测，标签通常为0（正常）或1（癫痫）。
    """
    # 检查结果是否可用且包含必要的数据
    if not file_results or 'file_names' not in file_results or not file_results['file_names']:
        print("\n无可用的文件级预测结果数据进行分析。")
        return
    
    print("\n==== 文件级预测结果摘要 ====")
    print("解读：这里显示的是模型对每一个独立的EEG片段（通常对应于输入数据中的一个文件或一个时间窗口）的预测结果。")
    
    file_names = file_results['file_names']
    subject_ids = file_results.get('subject_ids', ['N/A'] * len(file_names)) # 提供默认值防止缺少键
    predictions = file_results.get('predictions', [0] * len(file_names)) # 提供默认值
    probabilities = file_results.get('probabilities', [0.0] * len(file_names)) # 提供默认值
    
    # 基本统计
    total_files = len(file_names)
    # 确保 predictions 是可迭代的并且包含数字
    positive_count = sum(1 for p in predictions if p == 1) if predictions is not None else 0
    negative_count = total_files - positive_count
    
    print(f"总文件/片段数: {total_files}")
    if total_files > 0:
        print(f"预测为癫痫的片段: {positive_count} ({positive_count/total_files*100:.1f}%)")
        print(f"预测为正常的片段: {negative_count} ({negative_count/total_files*100:.1f}%)")
    else:
         print("没有找到文件级预测结果数据。")
    
    # 显示文件预测样本 (前5个)
    if total_files > 0:
        print("\n文件预测示例 (前5个):")
        print("解读：")
        print("  - '受试者ID': 该片段所属的受试者标识符。")
        print("  - '文件名': 输入的原始数据文件名。")
        print("  - '预测': 模型对此片段的二分类预测结果（1代表癫痫，0代表正常）。")
        print("  - '概率': 模型预测此片段为癫痫的概率值 (0到1之间)。概率越接近1，模型越确定是癫痫。")
        table_data = []
        for i in range(min(5, total_files)):
            prediction_label = "癫痫 (1)" if predictions[i] == 1 else "正常 (0)"
            table_data.append([
                subject_ids[i] if i < len(subject_ids) else 'N/A',
                os.path.basename(file_names[i]) if i < len(file_names) else 'N/A',
                prediction_label,
                f"{probabilities[i]:.4f}" if i < len(probabilities) else 'N/A'
            ])
        
        print(tabulate(table_data, headers=["受试者ID", "文件名", "预测", "概率"], tablefmt="pretty"))
        
        # 如果文件超过5个，显示总数
        if total_files > 5:
            print(f"... 以及其他 {total_files - 5} 个片段")

def analyze_subject_results(subject_results):
    """
    分析并解读受试者级预测结果。
    受试者级结果是对同一个受试者的多个EEG片段预测结果的汇总。
    """
    if not subject_results:
        print("\n无可用的受试者级预测结果数据进行分析。")
        return
    
    print("\n==== 受试者级预测结果摘要 ====")
    print("解读：这里显示的是模型对每个受试者的汇总诊断结果。由于每个受试者通常有多个EEG片段，需要将这些片段的预测结果整合起来。")
    
    # 基本统计
    total_subjects = len(subject_results)
    # 确保 data.get('vote_result') 和 data.get('prob_result') 存在
    vote_positive = sum(1 for subj, data in subject_results.items() if data.get('vote_result') == 1)
    prob_positive = sum(1 for subj, data in subject_results.items() if data.get('prob_result') == 1)

    print(f"总受试者数: {total_subjects}")
    if total_subjects > 0:
        print(f"基于多数投票诊断为癫痫的受试者: {vote_positive} ({vote_positive/total_subjects*100:.1f}%)")
        print(f"基于平均概率诊断为癫痫的受试者: {prob_positive} ({prob_positive/total_subjects*100:.1f}%)")
    else:
        print("没有找到受试者级预测结果数据。")
    
    # 准备表格数据
    if total_subjects > 0:
        table_data = []
        for subj_id, data in subject_results.items():
            vote_result = "癫痫 (1)" if data.get('vote_result') == 1 else "正常 (0)"
            prob_result = "癫痫 (1)" if data.get('prob_result') == 1 else "正常 (0)"
            segments_count = data.get('segments_count', 0)
            positive_segments = data.get('positive_segments', 0)
            positive_pct = positive_segments / segments_count * 100 if segments_count > 0 else 0
            
            table_data.append([
                subj_id,
                f"{positive_segments}/{segments_count} ({positive_pct:.1f}%)",
                vote_result,
                f"{data.get('avg_probability', 0.0):.4f}", # 提供默认值
                prob_result
            ])
        
        print("\n受试者诊断结果:")
        print("解读：")
        print("  - '受试者ID': 受试者标识符。")
        print("  - '阳性片段': 该受试者总片段数中有多少被模型预测为癫痫（显示数量及百分比）。")
        print("  - '投票诊断': 基于该受试者所有片段的多数投票结果（如果预测为癫痫的片段数多于正常片段数，则诊断为癫痫）。")
        print("  - '平均概率': 该受试者所有片段预测为癫痫的平均概率。")
        print("  - '概率诊断': 基于受试者平均概率是否高于设定的阈值（通常是0.5或其他值）得出的诊断结果。")

        # 只显示部分受试者数据，如果太多的话
        display_limit = 10
        if len(table_data) > display_limit:
            print(tabulate(table_data[:display_limit],
                          headers=["受试者ID", "阳性片段", "投票诊断", "平均概率", "概率诊断"],
                          tablefmt="pretty"))
            print(f"... 以及其他 {len(table_data) - display_limit} 个受试者")
        else:
            print(tabulate(table_data,
                          headers=["受试者ID", "阳性片段", "投票诊断", "平均概率", "概率诊断"],
                          tablefmt="pretty"))

def visualize_results(results, output_dir):
    """
    创建简单的结果可视化。
    绘制受试者平均概率的分布图，帮助理解模型的整体置信度以及阈值的影响。
    """
    has_visualized = False
    
    # 只有在受试者结果可用时才绘制概率分布图
    if 'subject_results' in results and results['subject_results']:
        subject_data = results['subject_results']
        
        # 收集数据用于可视化
        avg_probs = []
        
        for subj_id, data in subject_data.items():
             # 确保avg_probability存在且是数字
            avg_prob = data.get('avg_probability')
            if avg_prob is not None:
                avg_probs.append(avg_prob)

        if avg_probs: # 只有当有数据时才绘制
            # 创建概率分布图
            plt.figure(figsize=(10, 6))
            plt.hist(avg_probs, bins=20, alpha=0.7, color='blue', edgecolor='black')
            
            # 尝试从受试者结果数据中获取阈值，否则使用默认值
            # 找任意一个受试者的结果来获取阈值（假设所有受试者使用相同的阈值）
            sample_data = next(iter(subject_data.values()), {})
            threshold = sample_data.get('threshold', 0.5)
            
            plt.axvline(threshold, color='red', linestyle='--', label=f'诊断阈值 ({threshold})')
            plt.xlabel('受试者平均预测概率')
            plt.ylabel('受试者数量')
            plt.title('受试者癫痫预测平均概率分布')
            plt.legend()
            plt.grid(alpha=0.3)
            
            # 保存图像
            plot_path = os.path.join(output_dir, 'probability_distribution.png')
            try:
                plt.savefig(plot_path)
                print(f"\n✓ 概率分布图已保存至: {plot_path}")
                print("解读：此图显示了所有受试者的平均预测概率分布。阈值线将受试者分为预测为正常和预测为癫痫两组。")
                has_visualized = True
            except Exception as e:
                 print(f"✗ 保存概率分布图时出错: {e}")
                 print(f"  错误信息: {e}")
            finally:
                 plt.close()
        else:
            print("\n没有足够的受试者平均概率数据进行可视化。")

    if not has_visualized:
        print("\n无法创建可视化，受试者级预测结果数据不足或出错。")

def export_to_excel(results, output_dir):
    """
    将结果导出到Excel文件。
    将文件级、受试者级和详细CSV数据分别保存到Excel的不同工作表中，方便离线查看和进一步分析。
    """
    excel_output_path = os.path.join(output_dir, 'prediction_results.xlsx')
    try:
        # 使用 xlsxwriter 引擎，因为它在某些情况下更稳定
        # 只有当至少有一个 DataFrame 可以写入时才创建 ExcelWriter
        can_write = False

        # 文件级预测结果表
        df_files = None
        if 'file_results' in results and results['file_results']:
            file_data = results['file_results']
            # 检查键是否存在且非空
            if 'file_names' in file_data and len(file_data['file_names']) > 0:
                df_files = pd.DataFrame({
                    '受试者ID': file_data.get('subject_ids', ['N/A'] * len(file_data['file_names'])),
                    '文件名': [os.path.basename(f) for f in file_data['file_names']],
                    # 确保 predictions 存在
                    '预测': ['癫痫 (1)' if p == 1 else '正常 (0)' for p in file_data.get('predictions', [0] * len(file_data['file_names']))],
                     # 确保 probabilities 存在
                    '概率': file_data.get('probabilities', [0.0] * len(file_data['file_names']))
                })
                can_write = True
            else:
                print("警告: 文件级预测结果数据结构异常或为空，无法导出到Excel。")

        # 受试者级预测结果表
        df_subjects = None
        if 'subject_results' in results and results['subject_results']:
            subject_data = results['subject_results']
            if subject_data: # 检查字典是否非空
                # 转换成DataFrame
                subj_records = []
                for subj_id, data in subject_data.items():
                    segments_count = data.get('segments_count', 0)
                    positive_segments = data.get('positive_segments', 0)
                    subj_records.append({
                        '受试者ID': subj_id,
                        '总片段数': segments_count,
                        '阳性片段数': positive_segments,
                        '阳性比例': positive_segments / segments_count if segments_count > 0 else 0,
                        '投票结果': '癫痫 (1)' if data.get('vote_result') == 1 else '正常 (0)',
                        '平均概率': data.get('avg_probability', 0.0),
                        '概率结果': '癫痫 (1)' if data.get('prob_result') == 1 else '正常 (0)',
                        '阈值': data.get('threshold', 0.5)
                    })
                df_subjects = pd.DataFrame(subj_records)
                can_write = True
            else:
                print("警告: 受试者级预测结果数据为空，无法导出到Excel。")

        # 详细CSV数据 (已是DataFrame)
        df_csv = None
        if 'csv_data' in results and isinstance(results['csv_data'], pd.DataFrame):
             if not results['csv_data'].empty:
                df_csv = results['csv_data']
                # 确保 CSV 的列名与期望一致，并进行重命名以便于理解
                csv_columns_mapping = {
                    'Subject_ID': '受试者ID',
                    'File_Name': '文件名',
                    'Prediction': '预测 (0=正常, 1=癫痫)',
                    'Probability': '概率'
                }
                # 只保留在 CSV 中存在的列进行重命名
                df_csv.rename(columns={k: v for k, v in csv_columns_mapping.items() if k in df_csv.columns}, inplace=True)
                can_write = True
             else:
                print("警告: 详细CSV数据为空，无法导出到Excel。")

        # 如果有任何数据可以写入，则创建 ExcelWriter 并写入
        if can_write:
            writer = pd.ExcelWriter(excel_output_path, engine='xlsxwriter')
            if df_files is not None:
                 df_files.to_excel(writer, sheet_name='文件级预测', index=False)
            if df_subjects is not None:
                 df_subjects.to_excel(writer, sheet_name='受试者级预测', index=False)
            if df_csv is not None:
                 df_csv.to_excel(writer, sheet_name='详细预测数据', index=False)

            writer.close()
            print(f"\n✓ 结果已导出到Excel文件: {excel_output_path}")
            print("解读：打开此Excel文件，您可以在不同的工作表中查看文件级、受试者级和详细的预测结果，便于进一步的分析和报告。")
            return True
        else:
            print("\n没有可导出的有效数据到Excel。请确认预测结果文件已成功生成。")
            return False

    except Exception as e:
        print(f"✗ 导出到Excel时出错: {e}")
        print(f"  错误信息: {e}")
        print("  请确保您安装了 openpyxl 和 xlsxwriter 包 (pip install openpyxl xlsxwriter)")
        return False

def main():
    """主函数：加载、分析、可视化和导出预测结果。"""
    # 直接硬编码预测结果文件夹路径
    # 请确保这个路径是正确的！
    output_dir = r"outputs/predictions"
    
    print("\n" + "=" * 60)
    print("==== 癫痫预测结果分析工具 ====")
    print("=" * 60)
    print(f"正在分析目录：{output_dir}")
    print("此路径已硬编码在脚本中，如需更改请修改脚本。\n")
    print("-" * 60)
    
    # 检查目录是否存在
    if not os.path.exists(output_dir):
        print(f"错误: 指定的预测结果目录不存在 - {output_dir}")
        print("请确认路径是否正确，以及预测命令是否已成功运行。")
        print("\n分析终止。")
        print("-" * 60)
        return # 退出脚本
    
    # 加载结果
    results = load_prediction_results(output_dir)
    
    # 检查是否至少加载了一个结果文件
    if not results:
         print("\n未能成功加载任何预测结果文件。")
         print("请确认预测命令已成功运行并生成了 predictions.npy, subject_predictions.npy, 和 detailed_predictions.csv 文件在指定的目录中。")
         print("分析终止。")
         print("-" * 60)
         return # 退出脚本

    # 分析文件级结果
    # 即使只加载了 predictions.npy，也可以进行文件级分析
    if 'file_results' in results:
        analyze_file_results(results['file_results'])
    else:
        print("\n跳过文件级结果分析，因为文件未成功加载。")

    # 分析受试者级结果
    # 即使只加载了 subject_predictions.npy，也可以进行受试者级分析
    if 'subject_results' in results:
        analyze_subject_results(results['subject_results'])
    else:
        print("\n跳过受试者级结果分析，因为文件未成功加载。")
    
    # 生成可视化
    # 只有在成功加载受试者结果时才尝试可视化，因为概率分布图是基于受试者平均概率的
    if 'subject_results' in results:
         visualize_results(results, output_dir)
    else:
        print("\n跳过可视化，因为受试者级结果文件未成功加载。")
    
    # 导出到Excel
    # 即使只加载了部分文件，也尝试导出已加载的数据到Excel
    export_to_excel(results, output_dir)

    print("\n" + "=" * 60)
    print("==== 分析完成 ====")
    print("=" * 60)
    print(f"请检查控制台输出获取摘要，并查看预测结果目录 ({output_dir}) 以获取生成的图表和Excel文件。")
    print("-" * 60)


if __name__ == "__main__":
    main()